In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-03-26 12:11:51 +05:30)


In [2]:
import os
from pathlib import Path
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

time: 9.33 s (started: 2026-03-26 12:11:51 +05:30)


In [3]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 0 ns (started: 2026-03-26 12:12:00 +05:30)


In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"

time: 0 ns (started: 2026-03-26 12:12:00 +05:30)


In [5]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False 
)

Found 624 images belonging to 2 classes.
time: 93 ms (started: 2026-03-26 12:12:00 +05:30)


In [6]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-03-26 12:12:00 +05:30)


## Evalute Class_weight

In [7]:
model_class_weights = load_model("best_model_class_weights.h5")
result_class_weights = model_class_weights.evaluate(test_generator, verbose=1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 293ms/step - accuracy: 0.7324 - loss: 1.0950
time: 8.45 s (started: 2026-03-26 12:12:00 +05:30)


In [8]:
raw_preds_class_weights = model_class_weights.predict(test_generator).flatten()

best_thresh_class_weights, best_f1_class_weights = 0.5, 0

for t in np.arange(0.2, 0.8, 0.01):
    preds_class_weights = (raw_preds_class_weights > t).astype(int)
    f1_class_weights = f1_score(y_true, preds_class_weights)
    
    if f1_class_weights > best_f1_class_weights:
        best_f1_class_weights = f1_class_weights
        best_thresh_class_weights = t

print(f"Best threshold: {best_thresh_class_weights:.2f}, F1: {best_f1_class_weights:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 259ms/step
Best threshold: 0.79, F1: 0.8400
time: 6.81 s (started: 2026-03-26 12:12:09 +05:30)


In [9]:
y_pred_class_weights = (model_class_weights.predict(test_generator) > best_thresh_class_weights).astype(int)

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 260ms/step
time: 6.33 s (started: 2026-03-26 12:12:16 +05:30)


In [10]:
print("Accuracy0:", (result_class_weights[1]))

Accuracy0: 0.7323718070983887
time: 0 ns (started: 2026-03-26 12:12:22 +05:30)


In [11]:
print("Report0:\n", classification_report(y_true, y_pred_class_weights))

Report0:
               precision    recall  f1-score   support

           0       0.85      0.48      0.62       234
           1       0.75      0.95      0.84       390

    accuracy                           0.77       624
   macro avg       0.80      0.72      0.73       624
weighted avg       0.79      0.77      0.76       624

time: 15 ms (started: 2026-03-26 12:12:22 +05:30)


In [12]:
print("Unique predictions:", np.unique(y_pred_class_weights, return_counts=True))
print("True distribution: ", np.unique(y_true, return_counts=True))

Unique predictions: (array([0, 1]), array([133, 491]))
True distribution:  (array([0, 1], dtype=int32), array([234, 390]))
time: 0 ns (started: 2026-03-26 12:12:22 +05:30)


In [13]:
print("Report:\n",confusion_matrix(y_true, y_pred_class_weights))

Report:
 [[113 121]
 [ 20 370]]
time: 0 ns (started: 2026-03-26 12:12:22 +05:30)


## Evalute OverSampling

In [14]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-03-26 12:12:22 +05:30)


In [15]:
model_OverSampling = load_model("best_model_OverSampling.h5")
result_OverSampling = model_OverSampling.evaluate(test_generator, verbose=1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 257ms/step - accuracy: 0.6266 - loss: 1.7941
time: 7.31 s (started: 2026-03-26 12:12:22 +05:30)


In [16]:
test_generator.reset()
raw_preds_OverSampling = model_OverSampling.predict(test_generator).flatten()

best_thresh_OverSampling, best_f1_OverSampling = 0.5, 0

for t in np.arange(0.2, 0.8, 0.01):
    preds_OverSampling = (raw_preds_OverSampling > t).astype(int)
    f1_OverSampling = f1_score(y_true, preds_OverSampling)
    
    if f1_OverSampling > best_f1_OverSampling:
        best_f1_OverSampling = f1_OverSampling
        best_thresh_OverSampling = t

print(f"Best threshold: {best_thresh_OverSampling:.2f}, F1: {best_f1_OverSampling:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 263ms/step
Best threshold: 0.77, F1: 0.7730
time: 6.83 s (started: 2026-03-26 12:12:29 +05:30)


In [17]:
y_pred_OverSampling = (model_OverSampling.predict(test_generator) > best_thresh_OverSampling).astype(int)

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 264ms/step
time: 6.51 s (started: 2026-03-26 12:12:36 +05:30)


In [18]:
print("Accuracy0:", (result_OverSampling[1]))

Accuracy0: 0.6266025900840759
time: 16 ms (started: 2026-03-26 12:12:43 +05:30)


In [19]:
print("Report0:\n", classification_report(y_true, y_pred_OverSampling))

Report0:
               precision    recall  f1-score   support

           0       1.00      0.02      0.04       234
           1       0.63      1.00      0.77       390

    accuracy                           0.63       624
   macro avg       0.82      0.51      0.41       624
weighted avg       0.77      0.63      0.50       624

time: 0 ns (started: 2026-03-26 12:12:43 +05:30)


In [20]:
print("Unique predictions:", np.unique(y_pred_OverSampling, return_counts=True))
print("True distribution: ", np.unique(y_true, return_counts=True))

Unique predictions: (array([0, 1]), array([  5, 619]))
True distribution:  (array([0, 1], dtype=int32), array([234, 390]))
time: 16 ms (started: 2026-03-26 12:12:43 +05:30)


In [21]:
print("Report:\n",confusion_matrix(y_true, y_pred_OverSampling))

Report:
 [[  5 229]
 [  0 390]]
time: 15 ms (started: 2026-03-26 12:12:43 +05:30)


## Evalute K-Fold

In [22]:
model_fold1 = load_model("best_model_fold_1.h5")
model_fold2 = load_model("best_model_fold_2.h5")
model_fold3 = load_model("best_model_fold_3.h5")

time: 1.25 s (started: 2026-03-26 12:12:43 +05:30)


In [23]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-03-26 12:12:44 +05:30)


In [24]:
test_generator.reset()
raw_preds_fold1 = model_fold1.predict(test_generator).flatten()

test_generator.reset()
raw_preds_fold2 = model_fold2.predict(test_generator).flatten()

test_generator.reset()
raw_preds_fold3 = model_fold3.predict(test_generator).flatten()

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 215ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 216ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 216ms/step
time: 16.4 s (started: 2026-03-26 12:12:44 +05:30)


In [25]:
raw_preds_kfold = (raw_preds_fold1 + raw_preds_fold2 + raw_preds_fold3) / 3

time: 0 ns (started: 2026-03-26 12:13:00 +05:30)


In [26]:
def find_best_threshold(y_true, raw_preds, name="Model"):
    best_thresh = 0.5
    best_f1 = 0

    for t in np.arange(0.2, 0.8, 0.01):
        preds = (raw_preds > t).astype(int)
        f1 = f1_score(y_true, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print(f"{name} → Best Threshold: {best_thresh:.2f}, F1: {best_f1:.4f}")
    return best_thresh

time: 15 ms (started: 2026-03-26 12:13:00 +05:30)


In [27]:
t1 = find_best_threshold(y_true, raw_preds_fold1, "Fold 1")
t2 = find_best_threshold(y_true, raw_preds_fold2, "Fold 2")
t3 = find_best_threshold(y_true, raw_preds_fold3, "Fold 3")
t_ens = find_best_threshold(y_true, raw_preds_kfold, "Ensemble")

Fold 1 → Best Threshold: 0.80, F1: 0.9096
Fold 2 → Best Threshold: 0.74, F1: 0.9070
Fold 3 → Best Threshold: 0.76, F1: 0.8943
Ensemble → Best Threshold: 0.79, F1: 0.9075
time: 594 ms (started: 2026-03-26 12:13:01 +05:30)


In [28]:
preds_fold1 = (raw_preds_fold1 > t1).astype(int)
preds_fold2 = (raw_preds_fold2 > t2).astype(int)
preds_fold3 = (raw_preds_fold3 > t3).astype(int)
preds_kfold = (raw_preds_kfold > t_ens).astype(int)

time: 0 ns (started: 2026-03-26 12:13:01 +05:30)


In [29]:
def evaluate_model(y_true, preds, name="Model"):
    print(f"\n-------------{name}-------------")
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))
    
    print("\nClassification Report:")
    print(classification_report(y_true, preds))

time: 0 ns (started: 2026-03-26 12:13:01 +05:30)


In [30]:
evaluate_model(y_true, preds_fold1, "Fold 1")
evaluate_model(y_true, preds_fold2, "Fold 2")
evaluate_model(y_true, preds_fold3, "Fold 3")
evaluate_model(y_true, preds_kfold, "Ensemble (KFold Avg)")


-------------Fold 1-------------
Confusion Matrix:
[[196  38]
 [ 33 357]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       234
           1       0.90      0.92      0.91       390

    accuracy                           0.89       624
   macro avg       0.88      0.88      0.88       624
weighted avg       0.89      0.89      0.89       624


-------------Fold 2-------------
Confusion Matrix:
[[189  45]
 [ 29 361]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.81      0.84       234
           1       0.89      0.93      0.91       390

    accuracy                           0.88       624
   macro avg       0.88      0.87      0.87       624
weighted avg       0.88      0.88      0.88       624


-------------Fold 3-------------
Confusion Matrix:
[[174  60]
 [ 26 364]]

Classification Report:
              precision    recall  f1-score   sup